# Fase 3 — Semana 2: pipeline de preprocesamiento en clases

**Grupo 4 · MCDI500 · Encuesta Nacional de Salud 2016-2017**

En la Sumativa 1 dejamos listo un conjunto de 5.511 personas para
estudiar cómo se asocian edad, sexo, escolaridad, ingreso y zona con
cinco indicadores de riesgo cardiovascular: hipertensión, diabetes,
colesterol alto, índice de masa corporal y actividad física. Ese
preprocesamiento vivía en funciones sueltas dentro de un notebook.

En esta entrega reescribimos esos mismos pasos como clases que
comparten una interfaz común. El criterio de éxito es concreto: el
pipeline con clases debe entregar exactamente el mismo conjunto que
guardamos en la Fase 2.

| Sección | Contenido |
|---|---|
| 1 | Configuración y carga del conjunto elegible F1-F2 |
| 2 | Pipeline de preprocesamiento en clases |
| 3 | Verificación contra el resultado de la Fase 2 |
| 4 | Validación: caso normal, casos límite y excepciones |
| 5 | Eficiencia: tiempo y memoria |
| 6 | Patrón de diseño Strategy aplicado a la imputación |
| 7 | Arquitectura y conclusiones |

## 1. Configuración y carga del conjunto elegible F1-F2

Partimos del archivo filtrado por ponderador en la Fase 2, antes de
la limpieza. Así las clases tienen que reproducir todo el
preprocesamiento, y el resultado se puede comparar con el conjunto
final guardado en esa fase. Las columnas se agrupan según su rol en
el estudio: predictoras sociodemográficas, indicadores de riesgo y
variables del diseño muestral.

In [ ]:
RUTA_DATOS = "data/processed/ens_variables_f1f2.xlsx"
COLUMNA_ID = "IdEncuesta"

# Predictoras sociodemográficas
COLUMNAS_PREDICTORAS_CONTINUAS = ["Edad", "anos_estudio_MINSAL_1", "as27"]
COLUMNAS_PREDICTORAS_NOMINALES = ["Sexo", "Zona"]
COLUMNAS_PREDICTORAS_ORDINALES = ["as28"]

# Indicadores de riesgo cardiovascular (se analizan por separado)
COLUMNAS_RESULTADO_BINARIAS = ["HTA"]
COLUMNAS_RESULTADO_NOMINALES = ["di3", "dis2"]
COLUMNAS_RESULTADO_ORDINALES = ["GPAQ"]
COLUMNAS_RESULTADO_CONTINUAS = ["IMC"]

# Diseño muestral: se conservan sin transformar
COLUMNAS_DISENO_MUESTRAL = ["Fexp_F1F2p_Corr", "Conglomerado", "Estrato"]

SEMILLA = 2026

COLUMNAS_ESPERADAS = (
    [COLUMNA_ID]
    + COLUMNAS_PREDICTORAS_CONTINUAS + COLUMNAS_PREDICTORAS_NOMINALES
    + COLUMNAS_PREDICTORAS_ORDINALES + COLUMNAS_RESULTADO_BINARIAS
    + COLUMNAS_RESULTADO_NOMINALES + COLUMNAS_RESULTADO_ORDINALES
    + COLUMNAS_RESULTADO_CONTINUAS + COLUMNAS_DISENO_MUESTRAL
)
print("Columnas declaradas:", len(COLUMNAS_ESPERADAS))

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# La raíz se busca aquí porque es la que permite importar src/;
# por eso no puede venir desde el propio src/carga.py.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), None)
if RAIZ is None:
    raise FileNotFoundError("No se encontró la raíz del repositorio (.git).")
sys.path.append(str(RAIZ / "src"))

from carga import cargar_conjunto, perfilar
from transformador import Transformador

np.random.seed(SEMILLA)
print("pandas", pd.__version__, "· NumPy", np.__version__, "· semilla", SEMILLA)

In [ ]:
datos = cargar_conjunto(RAIZ / RUTA_DATOS, COLUMNAS_ESPERADAS)
perfil = perfilar(datos)
perfil[perfil["nulos"] > 0]

El conjunto tiene 5.520 personas. Los nulos se concentran en `as27`
(995), `GPAQ` (196), `anos_estudio_MINSAL_1` (47), `IMC` (37) y `HTA`
(9). La no respuesta de `as28` no aparece aquí porque está codificada
como -9999: el pipeline debe convertirla en nulo antes de imputar.

## 2. Pipeline de preprocesamiento en clases

Cada paso de limpieza de la Fase 2 se reescribe como una subclase de
`Transformador` (`src/transformador.py`). La clase base fija el orden
de uso: `ajustar()` calcula los parámetros con el conjunto de
referencia y `transformar()` los aplica sobre una copia, sin volver a
calcularlos. Cada subclase solo define qué calcula (`aprender()`) y
cómo lo usa (`aplicar()`). Los pasos concretos de imputación,
codificación y escalamiento se incorporan en las subsecciones
siguientes.

## 2.1 Preparar los datos antes de imputar

Antes de rellenar cualquier hueco hay que resolver dos cosas que en la
Fase 2 se hicieron a mano.

La primera son los códigos de no respuesta. En `as28` (tramo de ingreso
del hogar) hay 818 personas con el valor -9999, que pandas trata como
un ingreso más. `MarcadorNoRespuesta` los convierte en nulos, pero
antes deja anotado en `as28_no_responde` quién no respondió: no
declarar el ingreso puede tener un significado propio y no conviene
perderlo.

La segunda son las 9 personas sin diagnóstico de hipertensión (`HTA`).
Un diagnóstico no se puede estimar sin afirmar algo que nadie declaró,
así que `EliminadorFilasNulas` las quita en lugar de rellenarlas.

Las dos clases heredan de `Transformador`: solo definen qué calculan y
cómo lo aplican. El orden de uso (ajustar antes de transformar) lo
controla la clase base.

In [ ]:
from imputadores import (
    MarcadorNoRespuesta, EliminadorFilasNulas, ImputadorFlexible,
    PorMedia, PorMediana, PorModa, PorMedianaDeTramo, comparar_estrategias,
)

marcador = MarcadorNoRespuesta("as28")
marcado = marcador.ajustar_transformar(datos)
print(marcador)
print("Personas con código -9999 en as28 (antes):", int((datos["as28"] == -9999).sum()))
print("Nulos en as28 (después):", int(marcado["as28"].isna().sum()))
print("Marcadas en as28_no_responde:", int(marcado["as28_no_responde"].sum()))

eliminador = EliminadorFilasNulas("HTA")
sin_hta = eliminador.ajustar_transformar(marcado)
print(eliminador)
print(f"Filas: {len(marcado)} -> {len(sin_hta)} ({len(marcado) - len(sin_hta)} eliminadas por HTA nulo)")
print("as28_no_responde tras eliminar:", int(sin_hta["as28_no_responde"].sum()))

Se marcaron 818 personas, pero después de quitar las 9 filas sin `HTA`
quedan 816: dos de esas nueve también tenían -9999 en `as28`.

El orden importa. La bandera se crea antes de borrar los códigos, y las
filas se eliminan antes de imputar, porque en la Fase 2 las medianas se
calcularon sobre las 5.511 personas restantes. Si se imputara antes, los
valores de relleno cambiarían un poco y el resultado ya no coincidiría.

## 2.2 Imputación: una clase y varias formas de rellenar

`ImputadorFlexible` recibe la columna y una estrategia, y no sabe
rellenar por sí sola: le pide a la estrategia que calcule con qué y que
lo aplique. Las decisiones son las de la Fase 2:

| Columna | Estrategia | Por qué |
|---|---|---|
| `IMC`, `anos_estudio_MINSAL_1` | `PorMediana` | Distribuciones asimétricas |
| `GPAQ` | `PorModa` | Es ordinal (3,6 % de nulos): la moda conserva una categoría real |
| `as27` | `PorMedianaDeTramo` sobre `as28` | Solo se imputan las 177 personas que sí declararon su tramo; las 816 que no respondieron ninguno de los dos datos quedan sin valor |

In [ ]:
pasos_imputacion = [
    ImputadorFlexible("IMC", PorMediana()),
    ImputadorFlexible("anos_estudio_MINSAL_1", PorMediana()),
    ImputadorFlexible("as27", PorMedianaDeTramo("as28")),
    ImputadorFlexible("GPAQ", PorModa()),
]

imputado = sin_hta
for paso in pasos_imputacion:
    nulos_antes = int(imputado[paso.columna].isna().sum())
    imputado = paso.ajustar_transformar(imputado)   # misma llamada para todos
    nulos_despues = int(imputado[paso.columna].isna().sum())
    print(f"{paso.nombre:<40} nulos: {nulos_antes:>4} -> {nulos_despues}")

n_imputadas = int(imputado["as27_imputado"].sum())
n_sin_dato = int(imputado["as27"].isna().sum())
print("as27 imputadas por tramo:", n_imputadas)
print("as27 que quedan sin dato:", n_sin_dato)

assert n_imputadas == 177 and n_sin_dato == 816
assert imputado[["IMC", "anos_estudio_MINSAL_1", "GPAQ"]].isna().sum().sum() == 0
assert len(imputado) == len(sin_hta), "La imputación no debe cambiar el número de filas"
print("Verificado: los conteos coinciden con los de la Fase 2.")

Herencia, polimorfismo y encapsulamiento se ven así en este código:

- **Herencia:** los tres tipos de paso parten de `Transformador`; ninguno
  reescribió el manejo del estado.
- **Polimorfismo:** todos los pasos se usan con la misma llamada,
  `ajustar_transformar`, y `ImputadorFlexible` pide `calcular` y
  `rellenar` a la estrategia sin saber si es media, mediana, moda o
  por tramo.
- **Encapsulamiento:** lo aprendido queda guardado en `_parametros`; desde
  fuera solo se obtiene una copia, y usar un paso sin ajustarlo produce
  un error.

### 2.3 Comprobación del encapsulamiento

Las tres pruebas siguientes muestran el estado protegido. Primero se
transforma sin haber ajustado, algo que debe fallar (el error se captura
a propósito). Luego se usa el orden correcto, aprendiendo con una parte
de las personas y aplicando a la otra. Por último se intenta reemplazar
desde fuera lo que el paso aprendió.

In [ ]:
# 1) Transformar antes de ajustar: debe fallar, con un mensaje claro
paso_nuevo = ImputadorFlexible("IMC", PorMediana())
try:
    paso_nuevo.transformar(datos)
except RuntimeError as error:
    print("RuntimeError:", error)

# 2) Orden correcto: se aprende en entrenamiento y se aplica a prueba
entrenamiento = sin_hta.sample(frac=0.8, random_state=SEMILLA)
prueba = sin_hta.drop(index=entrenamiento.index)

paso = ImputadorFlexible("IMC", PorMediana()).ajustar(entrenamiento)
prueba_lista = paso.transformar(prueba)

print("\nMediana aprendida en entrenamiento   :", round(paso.parametros["valor"], 2))
print("Mediana propia del conjunto de prueba:", round(prueba["IMC"].median(), 2))
print("Nulos de IMC en prueba tras imputar  :", int(prueba_lista["IMC"].isna().sum()))

# 3) Lo aprendido no se puede reemplazar desde fuera
try:
    paso.parametros = {"valor": -1}
except AttributeError as error:
    print("\nAttributeError:", error)

Sin la comprobación previa, aplicar un paso antes de ajustarlo fallaría
de forma confusa o, peor, seguiría de largo sin avisar. Aquí el problema
aparece de inmediato y con una explicación.

Las dos medianas son parecidas, pero no idénticas. Lo importante es cuál
se usa: a la prueba se le aplica la del entrenamiento, así sus propios
datos no influyen en el relleno, aunque la diferencia sea pequeña.

### 2.4 Comprobación de la herencia

La herencia también se puede verificar en el propio código: el imputador
es a la vez un `ImputadorFlexible` y un `Transformador`, su cadena de
herencia muestra de dónde viene, y solo `aprender` y `aplicar` están
definidos en la clase hija. El resto de los métodos los aporta la clase
base.

In [ ]:
paso = ImputadorFlexible("IMC", PorMediana())

print("¿Es ImputadorFlexible?", isinstance(paso, ImputadorFlexible))
print("¿Es Transformador?    ", isinstance(paso, Transformador))
print("Cadena de herencia    :", [clase.__name__ for clase in ImputadorFlexible.__mro__])

print("\nDónde está definido cada método:")
for metodo in ["ajustar", "transformar", "ajustar_transformar", "aprender", "aplicar"]:
    origen = "ImputadorFlexible" if metodo in ImputadorFlexible.__dict__ else "Transformador"
    print(f"  {metodo:<22}{origen}")

## 3. Codificación de variables categóricas

Como parte del pipeline de preprocesamiento orientado a objetos, se incorpora la clase `CodificadorOneHot`, que hereda de la clase base `Transformador`.

Su objetivo es transformar las variables categóricas seleccionadas de la ENS en columnas binarias mediante **One-Hot Encoding**, manteniendo nombres semánticos asociados a las categorías originales.

Las variables consideradas son:

- `Sexo`
- `Zona`
- `di3`
- `dis2`

La implementación mantiene la misma interfaz utilizada por los demás componentes del pipeline: `ajustar()`, `transformar()` y `ajustar_transformar()`.

In [ ]:
from transformadores import (
    CodificadorOneHot,
    EscaladorEstandar,
    EliminadorColumna,
    ConvertidorEntero,
)

### 3.1 Categorías utilizadas

La codificación utiliza las categorías definidas para las variables seleccionadas de la ENS.

Además de realizar la transformación, `CodificadorOneHot` valida durante el ajuste que los códigos observados correspondan a categorías conocidas. De esta manera, un código no contemplado genera una excepción en lugar de ser procesado silenciosamente.

In [ ]:
variables_categoricas = ["Sexo", "Zona", "di3", "dis2"]

for columna in variables_categoricas:
    print(f"\n--- {columna} ---")
    print(
        imputado[columna]
        .value_counts(dropna=False)
        .sort_index()
    )

#### Definición de categorías

Las categorías utilizadas por `CodificadorOneHot` se definen explícitamente a partir del libro de códigos de la ENS 2016–2017 y no se infieren únicamente desde los valores observados en cada muestra.

Esta decisión permite mantener un esquema de salida estable. Por ejemplo, si una muestra contiene únicamente el código `1` para `Sexo`, el transformador conserva igualmente las columnas correspondientes a todas las categorías válidas definidas para esa variable.

Además, tanto durante el ajuste como durante la transformación se valida que los códigos observados pertenezcan al conjunto de categorías permitidas, evitando incorporar silenciosamente valores no reconocidos.

### 3.2 Aplicación y validación de `CodificadorOneHot`

Cada variable categórica se transforma utilizando una instancia independiente de `CodificadorOneHot`.

Durante `ajustar()`, el transformador valida los códigos presentes y almacena las categorías correspondientes. Posteriormente, `transformar()` genera una columna binaria por categoría y elimina la variable categórica original.

El proceso se aplica secuencialmente para mantener la lógica común definida por la clase base `Transformador`.

In [ ]:
df_codificado = imputado.copy()

codificadores = {}

for columna in variables_categoricas:
    codificador = CodificadorOneHot(columna)

    df_codificado = codificador.ajustar_transformar(
        df_codificado
    )

    codificadores[columna] = codificador

print(
    "Dimensiones antes de codificar:",
    imputado.shape
)

print(
    "Dimensiones después de codificar:",
    df_codificado.shape
)

In [ ]:
columnas_onehot = [
    columna
    for columna in df_codificado.columns
    if columna.startswith(("Sexo_", "Zona_", "di3_", "dis2_"))
]

print("Columnas generadas:")
for columna in columnas_onehot:
    print("-", columna)

In [ ]:
print("Valores únicos por columna:\n")

for columna in columnas_onehot:
    print(
        columna,
        sorted(df_codificado[columna].unique())
    )

### 3.3 Validación de la codificación

La transformación generó correctamente las variables binarias asociadas a las categorías de `Sexo`, `Zona`, `di3` y `dis2`.

Las columnas resultantes contienen exclusivamente valores `0` y `1`, mientras que las variables categóricas originales son retiradas del conjunto transformado.

La implementación permite además conservar la semántica de las categorías mediante nombres descriptivos, evitando utilizar únicamente los códigos numéricos originales de la ENS.

Las validaciones unitarias y los casos de excepción de este componente se encuentran implementados separadamente en `tests/test_transformadores.py`.

## 4. Escalamiento de variables numéricas

Como siguiente etapa del pipeline se incorpora `EscaladorEstandar`, una subclase de `Transformador` destinada a estandarizar variables numéricas.

Para una observación \(x\), la transformación aplicada corresponde a:

\[
z = \frac{x-\mu}{\sigma}
\]

donde:

- \(x\) es el valor original;
- \(\mu\) es la media calculada durante el ajuste;
- \(\sigma\) es la desviación estándar calculada durante el ajuste.

La separación entre `ajustar()` y `transformar()` permite almacenar los parámetros aprendidos y reutilizarlos posteriormente sobre nuevos datos.

### 4.1 Validación de `EscaladorEstandar`

Antes de aplicar una transformación numérica se verifica que las variables seleccionadas no contengan valores faltantes.

Esta comprobación es relevante porque el escalamiento corresponde a una etapa posterior al tratamiento de valores nulos dentro del pipeline de preprocesamiento.

In [ ]:
print("Estado de as27 después de la etapa de imputación:")
print("Registros totales:", len(imputado))
print("Valores disponibles:", imputado["as27"].notna().sum())
print("Valores nulos:", imputado["as27"].isna().sum())

print("\nResumen estadístico de as27:")
print(imputado["as27"].describe())

In [ ]:
columnas_ingreso = [
    columna for columna in imputado.columns
    if "as27" in columna.lower() or "as28" in columna.lower()
]

print("Columnas relacionadas con ingreso:")
print(columnas_ingreso)

### 4.2 Aplicación de `EscaladorEstandar`
El escalamiento se aplica después de la etapa de imputación.

`Edad` e `IMC` pueden escalarse sobre la totalidad de los registros disponibles. En el caso de `as27`, se conservan los valores faltantes definidos por la estrategia de imputación de la Fase 2.

La estrategia `PorMedianaDeTramo` imputa únicamente los casos en que existe información del tramo de ingreso (`as28`). Los participantes que no entregaron información suficiente mantienen `as27` como valor faltante, evitando introducir ingresos artificiales.

Por lo tanto, el escalamiento no modifica la política de tratamiento de valores faltantes definida previamente.

In [ ]:
df_escalado = imputado.copy()

variables_numericas = ["Edad", "IMC", "as27"]

escaladores = {}

for columna in variables_numericas:
    escalador = EscaladorEstandar(columna)

    df_escalado = escalador.ajustar_transformar(
        df_escalado
    )

    escaladores[columna] = escalador

print("Dimensiones:", df_escalado.shape)

print("\nNulos después del escalamiento:")
print(df_escalado[variables_numericas].isna().sum())

### 4.3 Validación estadística del escalamiento

Para verificar el funcionamiento de `EscaladorEstandar`, se comprueba que las variables transformadas presenten una media cercana a 0 y una desviación estándar cercana a 1.

En `as27`, las estadísticas se calculan sobre los valores disponibles. Los 816 valores faltantes definidos por la estrategia de imputación anterior se conservan y no intervienen en el cálculo.

In [ ]:
for columna in variables_numericas:
    media = df_escalado[columna].mean()
    desviacion = df_escalado[columna].std(ddof=0)

    print(f"\n{columna}")
    print("Media:", round(media, 10))
    print("Desviación estándar:", round(desviacion, 10))

### 4.4 Resultado del escalamiento

Las tres variables presentan una media aproximadamente igual a 0 y una desviación estándar igual a 1 después de la transformación, confirmando el funcionamiento esperado de `EscaladorEstandar`.

En `as27` se mantienen los 816 valores faltantes provenientes de la etapa anterior. El escalador transforma únicamente los valores disponibles y no modifica la estrategia de imputación definida previamente.

Esta separación de responsabilidades permite que cada componente del pipeline mantenga una función específica: los imputadores gestionan los valores faltantes y el escalador realiza exclusivamente la transformación numérica.

## 5. Evaluación de eficiencia

Además de validar el funcionamiento de los transformadores, se evalúa su eficiencia computacional mediante comparaciones con implementaciones de referencia.

Se consideran dos dimensiones:

- **Tiempo de ejecución:** permite comparar el costo temporal de cada implementación.
- **Memoria pico:** permite estimar la memoria utilizada durante la ejecución.

Las mediciones se realizan mediante las funciones reutilizables definidas en `src/medicion.py`.

Se comparan:

1. `EscaladorEstandar` frente a `StandardScaler` de scikit-learn.
2. `CodificadorOneHot` frente a `pandas.get_dummies`.

Además de las mediciones empíricas, se analiza la complejidad temporal y espacial de las soluciones.

In [ ]:
from medicion import (
    medir_tiempo,
    medir_memoria,
    medir_tiempo_timeit,
)

from sklearn.preprocessing import StandardScaler

In [ ]:
def escalar_propio(df, columna):
    escalador = EscaladorEstandar(columna)

    return escalador.ajustar_transformar(
        df[[columna]].copy()
    )


def escalar_sklearn(df, columna):
    escalador = StandardScaler()

    resultado = df[[columna]].copy()

    resultado[columna] = escalador.fit_transform(
        resultado[[columna]]
    ).ravel()

    return resultado

### 5.1 Equivalencia de resultados

Antes de comparar eficiencia, se verifica que ambas implementaciones produzcan resultados numéricamente equivalentes.

Esta comprobación evita comparar el rendimiento de algoritmos que realizan transformaciones diferentes.

In [ ]:
datos_edad = imputado[["Edad"]].copy()

resultado_propio = escalar_propio(
    datos_edad,
    "Edad"
)

resultado_sklearn = escalar_sklearn(
    datos_edad,
    "Edad"
)

np.testing.assert_allclose(
    resultado_propio["Edad"].to_numpy(),
    resultado_sklearn["Edad"].to_numpy(),
    rtol=1e-10,
    atol=1e-10,
)

print("Resultados equivalentes: EscaladorEstandar == StandardScaler")

In [ ]:
factores = [1, 10, 50, 100]

base_escalamiento = imputado[["Edad"]].copy()

resultados_escalamiento_timeit = []

for factor in factores:
    muestra = pd.concat(
        [base_escalamiento] * factor,
        ignore_index=True
    )

    tiempo_propio = medir_tiempo_timeit(
        escalar_propio,
        muestra,
        "Edad"
    )

    tiempo_sklearn = medir_tiempo_timeit(
        escalar_sklearn,
        muestra,
        "Edad"
    )

    resultados_escalamiento_timeit.append({
        "factor": factor,
        "filas": len(muestra),
        "propio_s": tiempo_propio,
        "sklearn_s": tiempo_sklearn
    })

resultados_escalamiento_timeit = pd.DataFrame(
    resultados_escalamiento_timeit
)

resultados_escalamiento_timeit

#### Análisis de resultados

Para evaluar el comportamiento temporal del escalamiento se utilizó `timeit` sobre conjuntos de tamaño creciente, obtenidos mediante la replicación controlada del conjunto de referencia. Se evaluaron tamaños entre 5.511 y 551.100 observaciones.

Ambas implementaciones presentan un incremento del tiempo de ejecución a medida que aumenta el número de registros, comportamiento compatible con una complejidad temporal lineal \(O(n)\), dado que el escalamiento requiere procesar cada observación de la variable.

`EscaladorEstandar` presentó menores tiempos de ejecución que `StandardScaler` en los tamaños evaluados. Sin embargo, esta diferencia no implica una menor complejidad algorítmica, ya que ambas implementaciones son \(O(n)\). Las diferencias observadas pueden estar asociadas a costos constantes y a las validaciones y operaciones internas realizadas por cada implementación.

En consecuencia, la implementación propia se mantiene principalmente por su integración con la arquitectura `Transformador`, su comportamiento controlado dentro del pipeline y su utilidad pedagógica, y no únicamente por la diferencia de tiempo observada.

### 5.2 Medición de memoria del escalamiento

Una vez comprobada la equivalencia y evaluado el comportamiento temporal mediante `timeit`, se complementa el análisis mediante la medición de memoria pico.

Para esta medición se utiliza `tracemalloc`, a través de la función reutilizable `medir_memoria` definida en `src/medicion.py`.

El objetivo es comparar la memoria administrada por Python durante la ejecución de `EscaladorEstandar` y `StandardScaler`. Esta medición debe interpretarse como una aproximación a las asignaciones de memoria gestionadas por Python y no como una medición de la memoria total utilizada por el proceso.

In [ ]:
factores = [1, 10, 50, 100]

base_escalamiento = imputado[["Edad"]].copy()

resultados_memoria_escalamiento = []

for factor in factores:
    muestra = pd.concat(
        [base_escalamiento] * factor,
        ignore_index=True
    )

    memoria_propia, _ = medir_memoria(
        escalar_propio,
        muestra,
        "Edad"
    )

    memoria_sklearn, _ = medir_memoria(
        escalar_sklearn,
        muestra,
        "Edad"
    )

    resultados_memoria_escalamiento.append({
        "factor": factor,
        "filas": len(muestra),
        "memoria_propia_kb": memoria_propia / 1024,
        "memoria_sklearn_kb": memoria_sklearn / 1024,
    })

resultados_memoria_escalamiento = pd.DataFrame(
    resultados_memoria_escalamiento
)

resultados_memoria_escalamiento

### 5.3 Análisis de eficiencia del escalamiento

La evaluación temporal realizada mediante `timeit` muestra que tanto `EscaladorEstandar` como `StandardScaler` incrementan su tiempo de ejecución a medida que aumenta el número de observaciones. Este comportamiento es consistente con una complejidad temporal lineal \(O(n)\), ya que ambas implementaciones deben procesar las observaciones de la variable para realizar el escalamiento.

En los tamaños evaluados, `EscaladorEstandar` presentó menores tiempos de ejecución que `StandardScaler`. Sin embargo, esta diferencia corresponde al rendimiento experimental observado y no implica una menor complejidad algorítmica, puesto que ambas alternativas presentan un crecimiento compatible con \(O(n)\).

La evaluación de memoria mediante `tracemalloc` complementa el análisis temporal utilizando los mismos factores de crecimiento del conjunto de datos. Esta medición representa la memoria pico administrada por Python durante cada ejecución y no la memoria total utilizada por el proceso.

Considerando la equivalencia numérica previamente verificada, el comportamiento temporal observado y la integración directa con la arquitectura basada en `Transformador`, se mantiene `EscaladorEstandar` como implementación del proyecto. La decisión no se fundamenta únicamente en diferencias de tiempo o memoria, sino también en su integración, control y mantenibilidad dentro del pipeline.

In [ ]:
def codificar_propio(df, columna):
    codificador = CodificadorOneHot(columna)

    return codificador.ajustar_transformar(
        df[[columna]].copy()
    )


def codificar_pandas(df, columna):
    resultado = pd.get_dummies(
        df[[columna]],
        columns=[columna],
        prefix=columna,
        dtype=int
    )

    if columna == "Sexo":
        resultado = resultado.rename(
            columns={
                "Sexo_1": "Sexo_Hombre",
                "Sexo_2": "Sexo_Mujer",
            }
        )

    return resultado

In [ ]:
datos_sexo = imputado[["Sexo"]].copy()

resultado_propio_ohe = codificar_propio(
    datos_sexo,
    "Sexo"
)

resultado_pandas_ohe = codificar_pandas(
    datos_sexo,
    "Sexo"
)

pd.testing.assert_frame_equal(
    resultado_propio_ohe,
    resultado_pandas_ohe
)

print(
    "Resultados equivalentes: "
    "CodificadorOneHot == pandas.get_dummies"
)

### 5.4 Evaluación de eficiencia de `CodificadorOneHot`

Para evaluar el segundo transformador desarrollado, se compara `CodificadorOneHot` con `pandas.get_dummies`.

Primero se verifica que ambas implementaciones generen una representación binaria equivalente. Posteriormente se comparan sus tiempos de ejecución y memoria pico para distintos tamaños de entrada.

### 5.5 Medición de eficiencia de la codificación

Una vez comprobada la equivalencia entre `CodificadorOneHot` y `pandas.get_dummies`, se evalúa el comportamiento temporal mediante `timeit` y la memoria pico mediante `tracemalloc`.

Para observar el comportamiento frente al crecimiento del conjunto de datos, se utiliza como base la variable `Sexo` del conjunto de referencia y se generan conjuntos de tamaño creciente mediante factores de replicación 1, 10, 50 y 100.

De esta forma, se evalúan conjuntos desde 5.511 hasta 551.100 observaciones. La variable mantiene dos categorías válidas, por lo que el número de categorías \(k\) permanece constante durante el experimento.

In [ ]:
factores = [1, 10, 50, 100]

base_codificacion = imputado[["Sexo"]].copy()

resultados_codificacion = []

for factor in factores:
    muestra = pd.concat(
        [base_codificacion] * factor,
        ignore_index=True
    )

    # Medición temporal con timeit
    tiempo_propio = medir_tiempo_timeit(
        codificar_propio,
        muestra,
        "Sexo"
    )

    tiempo_pandas = medir_tiempo_timeit(
        codificar_pandas,
        muestra,
        "Sexo"
    )

    # Medición de memoria con tracemalloc
    memoria_propia, _ = medir_memoria(
        codificar_propio,
        muestra,
        "Sexo"
    )

    memoria_pandas, _ = medir_memoria(
        codificar_pandas,
        muestra,
        "Sexo"
    )

    resultados_codificacion.append({
        "factor": factor,
        "filas": len(muestra),
        "tiempo_propio_s": tiempo_propio,
        "tiempo_pandas_s": tiempo_pandas,
        "memoria_propia_kb": memoria_propia / 1024,
        "memoria_pandas_kb": memoria_pandas / 1024,
    })

resultados_codificacion = pd.DataFrame(
    resultados_codificacion
)

resultados_codificacion

### 5.6 Análisis de eficiencia de la codificación

Las mediciones realizadas con `timeit` muestran que tanto `CodificadorOneHot` como `pandas.get_dummies` incrementan su tiempo de ejecución a medida que aumenta el número de observaciones.

La implementación propia puede expresarse con una complejidad temporal \(O(n \cdot k)\), donde \(n\) corresponde al número de observaciones y \(k\) al número de categorías. En este experimento, `Sexo` mantiene dos categorías válidas, por lo que \(k\) permanece constante y el crecimiento respecto de \(n\) se comporta de forma aproximadamente lineal.

Para 5.511 observaciones, `pandas.get_dummies` presentó un tiempo ligeramente menor. A partir de los tamaños mayores evaluados, las diferencias cambian y `CodificadorOneHot` presentó menores tiempos en este experimento. Estas diferencias corresponden al rendimiento empírico observado y no representan una diferencia en el orden de complejidad algorítmica.

La medición con `tracemalloc` muestra un aumento de la memoria pico a medida que crece el conjunto de datos. En los tamaños evaluados, `pandas.get_dummies` presentó un menor consumo de memoria administrada por Python que la implementación propia. Esta medición debe interpretarse como memoria gestionada por Python y no como memoria total del proceso.

Considerando la equivalencia previamente verificada, el comportamiento temporal y espacial observado y los requisitos arquitectónicos del proyecto, se mantiene `CodificadorOneHot`. La decisión se fundamenta principalmente en su integración con `Transformador`, la validación explícita de códigos y la generación de un esquema de categorías controlado y estable.

### 5.7 Decisión técnica adoptada

Las mediciones realizadas muestran que las implementaciones propias y las alternativas de referencia presentan comportamientos compatibles con un crecimiento lineal respecto del número de observaciones cuando el número de variables o categorías se mantiene constante.

En el escalamiento, `EscaladorEstandar` presentó menores tiempos de ejecución que `StandardScaler` en los tamaños evaluados. En la codificación, `CodificadorOneHot` presentó menores tiempos en los conjuntos de mayor tamaño, mientras que `pandas.get_dummies` mostró un menor consumo de memoria pico administrada por Python.

Por lo tanto, la decisión de mantener `EscaladorEstandar` y `CodificadorOneHot` no se fundamenta únicamente en cuál implementación obtiene el menor tiempo o consumo de memoria en una medición particular. Se consideran también los requisitos de diseño del proyecto.

Las implementaciones propias se integran directamente con la arquitectura basada en `Transformador`, mantienen una interfaz común de ajuste y transformación, permiten validar explícitamente los códigos admitidos y conservan un esquema estable de categorías definido previamente. Estas características favorecen la cohesión, trazabilidad y mantenibilidad del pipeline.

En consecuencia, se adoptan `EscaladorEstandar` y `CodificadorOneHot` como componentes del pipeline de preprocesamiento. Las implementaciones de `scikit-learn` y `pandas` se utilizan como referencias para verificar equivalencia funcional y comparar experimentalmente el comportamiento temporal y espacial.

> **Nota sobre la medición de memoria:** `tracemalloc` registra asignaciones de memoria administradas por Python. Por esta razón, los valores obtenidos se utilizan como evidencia comparativa entre implementaciones dentro del mismo entorno de ejecución y no como una medición absoluta de toda la memoria utilizada por el proceso.

### 5.8 Integración del pipeline de transformación

Una vez validados individualmente los componentes, se integran en un único flujo de transformación.

El proceso parte desde `imputado`, resultado de las etapas anteriores de preparación e imputación, y aplica secuencialmente:

1. eliminación de `IdEncuesta` y `FechaInicioF1`, variables que no forman parte del conjunto procesado final;
2. conversión de `HTA` y `GPAQ` a tipo entero;
3. codificación One-Hot de `Sexo`, `Zona`, `di3` y `dis2`;
4. escalamiento estándar de `Edad`, `IMC` y `as27`.

Todas estas operaciones utilizan transformadores compatibles con la interfaz común definida por `Transformador`, lo que permite mantener un flujo modular, trazable y extensible.


In [ ]:
df_transformado = imputado.copy()

# 1. Eliminar columnas que no forman parte del dataset procesado final
for columna in ["IdEncuesta", "FechaInicioF1"]:
    df_transformado = EliminadorColumna(
        columna
    ).ajustar_transformar(df_transformado)

# 2. Convertir variables categóricas/ordinales a entero
for columna in ["HTA", "GPAQ"]:
    df_transformado = ConvertidorEntero(
        columna
    ).ajustar_transformar(df_transformado)

# 3. Codificación de variables categóricas
for columna in variables_categoricas:
    codificador = CodificadorOneHot(columna)
    df_transformado = codificador.ajustar_transformar(
        df_transformado
    )

# 4. Escalamiento de variables numéricas
for columna in variables_numericas:
    escalador = EscaladorEstandar(columna)
    df_transformado = escalador.ajustar_transformar(
        df_transformado
    )

print("Dimensiones iniciales:", imputado.shape)
print("Dimensiones finales:", df_transformado.shape)

In [ ]:
# Se mantiene el número de observaciones
assert len(df_transformado) == len(imputado)

# Las variables no analíticas fueron eliminadas
assert "IdEncuesta" not in df_transformado.columns
assert "FechaInicioF1" not in df_transformado.columns

# HTA y GPAQ fueron convertidas a tipo entero
assert pd.api.types.is_integer_dtype(
    df_transformado["HTA"]
)
assert pd.api.types.is_integer_dtype(
    df_transformado["GPAQ"]
)

# GPAQ conserva las categorías ordinales esperadas
assert set(df_transformado["GPAQ"].unique()) == {1, 2, 3}

# Las variables categóricas originales fueron reemplazadas
for columna in variables_categoricas:
    assert columna not in df_transformado.columns

# Las columnas One-Hot contienen únicamente 0 y 1
columnas_onehot = [
    columna
    for columna in df_transformado.columns
    if columna.startswith(
        ("Sexo_", "Zona_", "di3_", "dis2_")
    )
]

for columna in columnas_onehot:
    assert set(
        df_transformado[columna].dropna().unique()
    ).issubset({0, 1})

# Verificación estadística del escalamiento
for columna in variables_numericas:
    valores = df_transformado[columna].dropna()

    assert abs(valores.mean()) < 1e-10
    assert abs(
        valores.std(ddof=0) - 1
    ) < 1e-10

# Dimensión final esperada
assert df_transformado.shape == (5511, 23)

print("Pipeline integrado validado correctamente.")
print("Dimensiones finales:", df_transformado.shape)

In [ ]:
# Comparación estructural con el dataset procesado de referencia

ruta_referencia = RAIZ / "data" / "processed" / "ens_procesado.csv"

df_referencia = pd.read_csv(ruta_referencia)

columnas_transformado = set(df_transformado.columns)
columnas_referencia = set(df_referencia.columns)

faltantes = columnas_referencia - columnas_transformado
adicionales = columnas_transformado - columnas_referencia

print("Dimensiones pipeline F3:", df_transformado.shape)
print("Dimensiones referencia:", df_referencia.shape)

print("\nColumnas faltantes:")
print(faltantes)

print("\nColumnas adicionales:")
print(adicionales)

assert df_transformado.shape == df_referencia.shape
assert columnas_transformado == columnas_referencia

print("\nEstructura final compatible con ens_procesado.csv.")

#### Validación estructural del pipeline

Como verificación de integración con el preprocesamiento desarrollado en la Fase 2, se comparó la estructura obtenida mediante los transformadores de la Fase 3 con `ens_procesado.csv`, utilizado como conjunto procesado de referencia.

El pipeline conserva las 5.511 observaciones y genera 23 variables finales. Asimismo, no se detectaron columnas faltantes ni adicionales respecto del conjunto de referencia.

Esta comprobación permite verificar que la refactorización del preprocesamiento mediante componentes modulares mantiene la estructura esperada del dataset, incorporando explícitamente la eliminación de variables no analíticas, el casting de `HTA` y `GPAQ`, la codificación One-Hot y el escalamiento de las variables numéricas seleccionadas.

## 6. Patrón de diseño Strategy aplicado a la imputación

Para `as27` había varias formas razonables de rellenar los nulos. Con
Strategy, cada forma es una clase con los mismos dos métodos
(`calcular` y `rellenar`), e `ImputadorFlexible` la recibe como
parámetro. Así se pueden comparar alternativas cambiando solo el
objeto que se entrega. Se comparan tres sobre `as27`: media, mediana
y mediana por tramo de `as28`.

In [ ]:
estrategias_as27 = [PorMedia(), PorMediana(), PorMedianaDeTramo("as28")]
comparacion = comparar_estrategias(sin_hta, "as27", estrategias_as27)
comparacion

La media y la mediana rellenan los 993 nulos y achican la dispersión
alrededor de un 9 %, porque llenan también a las 816 personas que no
declararon ingreso. La mediana por tramo casi no la altera (-0,70 %),
pero solo rellena 177: las otras 816 quedan sin valor.

La comparación no es de igual contra igual. Se elige la mediana por
tramo porque usa un dato que la persona sí entregó (su tramo) y evita
inventar un ingreso para quienes no respondieron nada. Su límite es que
esa no respuesta podría no ser aleatoria, y eso sigue como limitación
para la interpretación de los resultados.

La comparación anterior no requirió modificar `ImputadorFlexible`. La
celda siguiente lo muestra de forma directa: la misma clase se usa con
las tres estrategias y solo cambia el objeto que recibe.

In [ ]:
for estrategia in estrategias_as27:
    paso = ImputadorFlexible("as27", estrategia)
    resultado = paso.ajustar_transformar(sin_hta)
    print(f"{estrategia.etiqueta:<20} nulos restantes en as27: {int(resultado['as27'].isna().sum())}")

**Qué habría pasado sin el patrón.** En la Fase 2, cada alternativa
para `as27` significó una función distinta o una rama `if` dentro de
`imputar_nulos_numericos`. Sumar la mediana por tramo habría obligado
a modificar esa función y el código que la llama, con el riesgo de
alterar la imputación de otras columnas.

**Por qué Strategy y no otro patrón.** El problema de esta parte es
tener varias formas intercambiables de hacer lo mismo sobre una
columna. Factory, Observer y Singleton resuelven problemas distintos
(decidir qué objeto construir, registrar eventos o compartir una única
configuración) que no aparecen en esta etapa del proyecto.